# Classifier Comparison – H-optimus-1 Edition

Compares **Zero-Shot (CONCH)**, **Few-Shot Linear Probe / Logistic Regression (CONCH)**, and **ANN (H-optimus-1)** classifiers on the CRC-100K test split and exports a colour-coded Excel report with embedded patch thumbnails.

> **Edit the `CFG` block in Cell 3 before running.**


In [46]:
import os, io, json, random
from pathlib import Path

import numpy as np
import torch
import timm
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from huggingface_hub import login
from sklearn.linear_model import LogisticRegression
import joblib

# CONCH imports  (zero-shot + few-shot)
from conch.open_clip_custom import create_model_from_pretrained

# openpyxl  (Excel report)
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

print("All imports OK ✅")

All imports OK ✅


## Configuration
Edit paths and parameters here.

In [47]:
import os
from dotenv import load_dotenv
load_dotenv()
# ── CONFIGURE ALL PATHS HERE ────────────────────────────────────────────────
CFG = dict(

    # ── Folders ──────────────────────────────────────────────────────────────
    # Train and test images live in DIFFERENT folders.
    # Each folder must contain one sub-folder per class.
    train_image_folder = r"D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM",          # training patches
    test_image_folder  = r"D:\Aamir Gulzar\dataset\CRC100K\CRC-VAL-HE-7K",  # test patches

    # ── Prompts file (class names used by Zero-Shot & Few-Shot) ──────────
    prompt_file = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\config_files\Custom_prompts_all_per_class.json",

    # ── Pre-computed Zero-Shot CONCH weights ─────────────────────────────
    zs_weights_path = ("D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\classifier_weights\Conch_zeroshot_weights.pt"),

    # ── CONCH feature caches ──────────────────────────────────────────────
    # conch_train_cache : CONCH features of TRAIN images → used to fit the LR
    # conch_test_cache  : CONCH features of TEST  images → used by ZS + LR inference
    conch_checkpoint = r"checkpoints\conch\pytorch_model.bin",
    conch_train_cache = (r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier"
                         r"\slide_aggregation\caption_generation\Results"
                         r"\Few_shot_features\conch_train_features.pt"),
    conch_test_cache  = (r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier"
                         r"\slide_aggregation\caption_generation\Results"
                         r"\Few_shot_features\conch_test_features.pt"),

    # ── H-optimus-1 – features for BOTH folders (used by ANN for inference) ─
    hoptimus_model_id    = "hf-hub:bioptimus/H-optimus-1",
    hoptimus_train_cache = r"Cache\hoptimus_train_features.pt",  # patches from train_image_folder
    hoptimus_test_cache  = r"Cache\hoptimus_test_features.pt",   # patches from test_image_folder
    hf_token             = os.environ.get("HF_TOKEN", ""),          # paste your HuggingFace token here if needed
    force_extract        = False,       # True → re-extract even if cache already exists

    # ── ANN: saved with torch.save(model), trained on H-optimus-1 features
    ann_model_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\models\ann_crc100k_Hoptimus.pth",

    # ── Logistic Regression (Few-Shot Linear Probe) ───────────────────────
    # Trained on CONCH train features; saved so you can reuse it.
    lr_model_path  = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_aggregation\caption_generation\Results\Few_Shot_Evaluation\180_trainset\few_shot_logistic_regression.pkl",

    # ── Split params: must match your original few-shot script exactly ───
    n_train     = 180,
    random_seed = 42,

    # ── How many examples to put in the report ───────────────────────────
    target_diff = 60,   # patches where ZS != ANN
    target_same = 20,   # patches where ZS == ANN

    # ── Output ───────────────────────────────────────────────────────────
    output_xlsx = r"Results\classifier_comparison_report.xlsx",
    thumb_size  = (125, 125),   # thumbnail size (px) embedded in Excel

    # ── TEST_ONLY: set to 1 to skip train cache and run inference on test set only ──
    # When 1: train arrays are never loaded, n_train=0, N=n_test, index arithmetic
    # maps global_idx directly to the test arrays (no subtraction needed).
    test_only   = 0,
)

# ── ANN class order ──────────────────────────────────────────────────────
# The ANN was trained on folder sub-directory names (e.g. 'ADI', 'BACK', …).
# These MUST match the keys of 'classnames' in the prompt JSON in the exact
# order the ANN saw them during training.  They are used as the single source
# of truth for BOTH the CONCH classifiers and the ANN — eliminating any
# label-index mismatch.

# Verify this order matches your ANN training script before running!
ANN_CLASS_ORDER = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

print("CFG loaded ✅")

CFG loaded ✅


## Dataset Helpers

In [48]:
def build_df_from_subfolders(image_folder, classes_list):
    records, exts = [], {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    for cls in classes_list:
        cls_dir = os.path.join(image_folder, cls)
        if not os.path.isdir(cls_dir):
            print(f"  [WARN] class folder not found: {cls_dir}")
            continue
        for fname in sorted(os.listdir(cls_dir)):
            if Path(fname).suffix.lower() in exts:
                records.append({
                    "patch_name": fname,
                    "label":      cls,
                    "full_path":  os.path.join(cls_dir, fname),
                })
    df = pd.DataFrame(records)
    print(f"  {len(df)} patches across {df['label'].nunique()} classes")
    return df


def split_train_test(df, classes_list, n_train_total, random_seed=42):
    """Exact replica of the original split logic so indices align with CONCH caches."""
    rng = np.random.default_rng(random_seed)
    per_class = n_train_total // len(classes_list)
    train_indices = []
    for cls in classes_list:
        idx = df.index[df["label"] == cls].tolist()
        rng.shuffle(idx)
        train_indices.extend(idx[:min(per_class, len(idx))])
    mask = df.index.isin(train_indices)
    return df[mask].reset_index(drop=True), df[~mask].reset_index(drop=True)

print("Dataset helpers defined ✅")

Dataset helpers defined ✅


## H-optimus-1 Feature Extractor

In [49]:
def build_hoptimus_transform():
    """
    Official H-optimus-1 preprocessing:
      – Resize to 224×224
      – ToTensor
      – Normalize with H-optimus-1 mean/std
    """
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.707223, 0.578729, 0.703617),
            std =(0.211883, 0.230117, 0.177517),
        ),
    ])


def load_hoptimus_model(model_id, hf_token, device):
    """Load H-optimus-1 via timm, log in to HuggingFace if a token is supplied."""
    if hf_token:
        login(token=hf_token)
    print(f"  Loading H-optimus-1 model ({model_id}) …")
    model = timm.create_model(
        model_id,
        pretrained=True,
        init_values=1e-5,
        dynamic_img_size=False,
    ).to(device)
    model.eval()
    return model


def extract_hoptimus_features(df, model_id, hf_token, cache_path, device,
                               force=False, _model=None):
    """
    Extract CLS-token embeddings from H-optimus-1 for every patch in df.
    Results are cached to `cache_path` so subsequent runs are instant.
    Pass _model to reuse an already-loaded model (avoids loading twice).
    Returns (features [N, D], labels [N], paths [N], names [N]).
    """
    if not force and os.path.exists(cache_path):
        print(f"  Loading cached H-optimus-1 features: {cache_path}")
        cache = torch.load(cache_path, weights_only=False)
        return cache["features"], cache["labels"], cache["paths"], cache["names"]

    model     = _model if _model is not None else load_hoptimus_model(model_id, hf_token, device)
    transform = build_hoptimus_transform()

    features, labels, paths, names = [], [], [], []
    with torch.inference_mode():
        for _, row in tqdm(df.iterrows(), total=len(df),
                           desc="Extracting H-optimus-1 features"):
            try:
                image  = Image.open(row["full_path"]).convert("RGB")
                tensor = transform(image).unsqueeze(0).to(device)   # [1, 3, 224, 224]
                emb    = model(tensor)                               # [1, D] CLS token
                emb    = emb / emb.norm(dim=-1, keepdim=True)       # L2 normalise
                features.append(emb.cpu().squeeze(0))
                labels.append(row["label"])
                paths.append(row["full_path"])
                names.append(row["patch_name"])
            except Exception as e:
                print(f"  [WARN] skipping {row['full_path']}: {e}")

    X = torch.stack(features)
    y = np.array(labels)
    os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
    torch.save({"features": X, "labels": y,
                "paths": paths, "names": names}, cache_path)
    print(f"  Saved cache → {cache_path}")

    # Only delete the model if we created it here (not if caller passed _model)
    if _model is None:
        del model; torch.cuda.empty_cache()
    return X, y, paths, names

print("H-optimus-1 extractor defined ✅")

H-optimus-1 extractor defined ✅


## Few-Shot Linear Probe & Excel Report Helpers

In [50]:
def fit_linear_probe(X_train_np, y_train, random_seed=42):
    """Fit a balanced Logistic Regression on CONCH train features."""
    clf = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=random_seed,
    )
    clf.fit(X_train_np, y_train)
    return clf


# ── Excel styling constants ──────────────────────────────────────────────────
FILL_DIFF = PatternFill("solid", fgColor="FFD0D0")
FILL_SAME = PatternFill("solid", fgColor="D0FFD0")
FILL_BONUS= PatternFill("solid", fgColor="EFEFEF")
FILL_HDR  = PatternFill("solid", fgColor="1F3864")
FILL_ZS   = PatternFill("solid", fgColor="E8F4FD")   # light blue for ZS-caption sheet
FONT_HDR  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
FONT_BODY = Font(name="Arial", size=9)
FONT_OK   = Font(name="Arial", size=9, bold=True, color="1A7A1A")
FONT_BAD  = Font(name="Arial", size=9, bold=True, color="BB1111")
ALIGN_C   = Alignment(horizontal="center", vertical="center", wrap_text=True)
ALIGN_L   = Alignment(horizontal="left",   vertical="center", wrap_text=True)
_SIDE     = Side(style="thin", color="BBBBBB")
BORDER    = Border(left=_SIDE, right=_SIDE, top=_SIDE, bottom=_SIDE)

HEADERS = [
    ("#",                  5),
    ("Patch Image",       20),
    ("Filename",          30),
    ("True Label",        16),
    ("Zero-Shot Pred",    16),
    ("ANN Pred",          16),
    ("Match?",            10),
    ("Few-Shot Pred",     18),
    ("ZS Confidence",     14),
    ("ANN Confidence",    14),
    ("FS Confidence",     14),
]

ZS_CAPTION_HEADERS = [
    ("#",                  5),
    ("Patch Image",       20),
    ("Filename",          30),
    ("True Label",        16),
    ("ZS Caption",        20),
    ("ANN Pred",          16),
    ("FS Pred",           16),
    ("ZS Confidence",     14),
    ("ANN Confidence",    14),
    ("FS Confidence",     14),
]


def _cell(ws, row, col, value, font=None, fill=None, align=None):
    c = ws.cell(row=row, column=col, value=value)
    if font:  c.font      = font
    if fill:  c.fill      = fill
    if align: c.alignment = align
    c.border = BORDER
    return c


def _write_sheet(wb, sheet_title, subset, thumb_size, first=False):
    ws = wb.active if first else wb.create_sheet()
    ws.title = sheet_title
    for col, (h, w) in enumerate(HEADERS, 1):
        _cell(ws, 1, col, h, font=FONT_HDR, fill=FILL_HDR, align=ALIGN_C)
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[1].height = 22
    ROW_H = thumb_size[1] + 10
    for i, r in enumerate(subset):
        row   = i + 2
        ws.row_dimensions[row].height = ROW_H
        fill  = FILL_SAME if r["match"] else FILL_DIFF
        f_cls = FONT_OK   if r["match"] else FONT_BAD
        _cell(ws, row, 1,  i + 1,                FONT_BODY, fill,       ALIGN_C)
        _cell(ws, row, 3,  r["filename"],         FONT_BODY, fill,       ALIGN_L)
        _cell(ws, row, 4,  r["true_label"],       FONT_BODY, fill,       ALIGN_C)
        _cell(ws, row, 5,  r["zs_pred"],          f_cls,     fill,       ALIGN_C)
        _cell(ws, row, 6,  r["ann_pred"],         f_cls,     fill,       ALIGN_C)
        _cell(ws, row, 7,
              "✓ SAME" if r["match"] else "✗ DIFF",
              f_cls, fill, ALIGN_C)
        _cell(ws, row, 8,  r["fs_pred"],           FONT_BODY, FILL_BONUS, ALIGN_C)
        _cell(ws, row, 9,  f"{r['zs_conf']:.1%}",  FONT_BODY, fill,    ALIGN_C)
        _cell(ws, row, 10, f"{r['ann_conf']:.1%}", FONT_BODY, fill,    ALIGN_C)
        _cell(ws, row, 11, f"{r['fs_conf']:.1%}",  FONT_BODY, FILL_BONUS, ALIGN_C)
        try:
            img = Image.open(r["full_path"]).convert("RGB")
            img.thumbnail(thumb_size, Image.LANCZOS)
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            buf.seek(0)
            ws.add_image(XLImage(buf), f"B{row}")
        except Exception as e:
            ws.cell(row=row, column=2, value=f"[img err: {e}]")
    return ws


def _write_zs_caption_sheet(wb, sheet_title, subset, thumb_size):
    """Write a sheet showing ZS-caption breakdown (14 captions)."""
    ws = wb.create_sheet()
    ws.title = sheet_title
    for col, (h, w) in enumerate(ZS_CAPTION_HEADERS, 1):
        _cell(ws, 1, col, h, font=FONT_HDR, fill=FILL_HDR, align=ALIGN_C)
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[1].height = 22
    ROW_H = thumb_size[1] + 10
    for i, r in enumerate(subset):
        row = i + 2
        ws.row_dimensions[row].height = ROW_H
        _cell(ws, row, 1,  i + 1,                    FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 3,  r["filename"],             FONT_BODY, FILL_ZS,   ALIGN_L)
        _cell(ws, row, 4,  r["true_label"],           FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 5,  r["zs_pred"],              FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 6,  r["ann_pred"],             FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 7,  r["fs_pred"],              FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 8,  f"{r['zs_conf']:.1%}",  FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 9,  f"{r['ann_conf']:.1%}", FONT_BODY, FILL_ZS,   ALIGN_C)
        _cell(ws, row, 10, f"{r['fs_conf']:.1%}",  FONT_BODY, FILL_ZS,   ALIGN_C)
        try:
            img = Image.open(r["full_path"]).convert("RGB")
            img.thumbnail(thumb_size, Image.LANCZOS)
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            buf.seek(0)
            ws.add_image(XLImage(buf), f"B{row}")
        except Exception as e:
            ws.cell(row=row, column=2, value=f"[img err: {e}]")
    return ws


def write_excel(diff_rows, same_rows, zs_caption_rows, output_path, thumb_size,
                ann_class_order, zs_class_order):
    """
    Write the full Excel report with four sheets:
      1. DIFFERENT (ZS≠ANN)    — 5-10 samples per true class (9 classes)
      2. SAME (ZS=ANN)         — 2-3 samples per true class (9 classes)
      3. ZS Captions           — 5-10 per ZS-assigned caption (14 captions), sorted
      4. Summary               — stats for selected patches
    """
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    wb = Workbook()

    # ── Sheet 1: DIFFERENT ──────────────────────────────────────────────────
    # Sort by true_label so all 9 classes appear in order
    diff_sorted = sorted(diff_rows, key=lambda r: ann_class_order.index(r["true_label"])
                         if r["true_label"] in ann_class_order else 999)
    _write_sheet(wb, "DIFFERENT (ZS≠ANN)", diff_sorted, thumb_size, first=True)

    # ── Sheet 2: SAME ───────────────────────────────────────────────────────
    same_sorted = sorted(same_rows, key=lambda r: ann_class_order.index(r["true_label"])
                         if r["true_label"] in ann_class_order else 999)
    _write_sheet(wb, "SAME (ZS=ANN)", same_sorted, thumb_size, first=False)

    # ── Sheet 3: ZS Captions ────────────────────────────────────────────────
    # Sort by ZS caption (zs_pred) in ZS_CLASS_ORDER order
    zs_sorted = sorted(zs_caption_rows,
                       key=lambda r: zs_class_order.index(r["zs_pred"])
                       if r["zs_pred"] in zs_class_order else 999)
    _write_zs_caption_sheet(wb, "ZS Captions (14 classes)", zs_sorted, thumb_size)

    # ── Sheet 4: Summary ────────────────────────────────────────────────────
    ws_s = wb.create_sheet("Summary")
    ws_s["A1"] = "Classifier Comparison – Summary"
    ws_s["A1"].font = Font(bold=True, size=13, name="Arial")

    # Top-level counts
    for i, (lbl, val) in enumerate([
        ("Total DIFFERENT patches in report",  len(diff_sorted)),
        ("Total SAME patches in report",        len(same_sorted)),
        ("Total ZS-caption patches in report",  len(zs_sorted)),
        ("Grand total patches",                 len(diff_sorted) + len(same_sorted)),
    ], 2):
        ws_s.cell(row=i, column=1, value=lbl).font = FONT_BODY
        ws_s.cell(row=i, column=2, value=val).font = FONT_BODY

    # ── Per-class breakdown: DIFFERENT sheet ────────────────────────────────
    row_cursor = 8
    ws_s.cell(row=row_cursor, column=1,
              value="Per-class breakdown — DIFFERENT (ZS≠ANN)").font = Font(bold=True, name="Arial")
    row_cursor += 1
    for col, h in enumerate(["True Class", "Count", "ZS Conf (avg)", "ANN Conf (avg)", "FS Conf (avg)"], 1):
        c = ws_s.cell(row=row_cursor, column=col, value=h)
        c.font = FONT_HDR; c.fill = FILL_HDR; c.alignment = ALIGN_C
        ws_s.column_dimensions[get_column_letter(col)].width = 20
    row_cursor += 1
    from collections import defaultdict
    diff_by_class = defaultdict(list)
    for r in diff_sorted:
        diff_by_class[r["true_label"]].append(r)
    for cls in ann_class_order:
        recs = diff_by_class.get(cls, [])
        cnt  = len(recs)
        zsc  = f"{sum(r['zs_conf'] for r in recs)/cnt:.1%}" if cnt else "—"
        ac   = f"{sum(r['ann_conf'] for r in recs)/cnt:.1%}" if cnt else "—"
        fc   = f"{sum(r['fs_conf'] for r in recs)/cnt:.1%}" if cnt else "—"
        ws_s.cell(row=row_cursor, column=1, value=cls).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=2, value=cnt).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=3, value=zsc).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=4, value=ac).font  = FONT_BODY
        ws_s.cell(row=row_cursor, column=5, value=fc).font  = FONT_BODY
        row_cursor += 1

    # ── Per-class breakdown: SAME sheet ─────────────────────────────────────
    row_cursor += 1
    ws_s.cell(row=row_cursor, column=1,
              value="Per-class breakdown — SAME (ZS=ANN)").font = Font(bold=True, name="Arial")
    row_cursor += 1
    for col, h in enumerate(["True Class", "Count", "ZS Conf (avg)", "ANN Conf (avg)", "FS Conf (avg)"], 1):
        c = ws_s.cell(row=row_cursor, column=col, value=h)
        c.font = FONT_HDR; c.fill = FILL_HDR; c.alignment = ALIGN_C
    row_cursor += 1
    same_by_class = defaultdict(list)
    for r in same_sorted:
        same_by_class[r["true_label"]].append(r)
    for cls in ann_class_order:
        recs = same_by_class.get(cls, [])
        cnt  = len(recs)
        zsc  = f"{sum(r['zs_conf'] for r in recs)/cnt:.1%}" if cnt else "—"
        ac   = f"{sum(r['ann_conf'] for r in recs)/cnt:.1%}" if cnt else "—"
        fc   = f"{sum(r['fs_conf'] for r in recs)/cnt:.1%}" if cnt else "—"
        ws_s.cell(row=row_cursor, column=1, value=cls).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=2, value=cnt).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=3, value=zsc).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=4, value=ac).font  = FONT_BODY
        ws_s.cell(row=row_cursor, column=5, value=fc).font  = FONT_BODY
        row_cursor += 1

    # ── Per-ZS-caption breakdown ─────────────────────────────────────────────
    row_cursor += 1
    ws_s.cell(row=row_cursor, column=1,
              value="Per-caption breakdown — ZS Captions (14 classes)").font = Font(bold=True, name="Arial")
    row_cursor += 1
    for col, h in enumerate(["ZS Caption", "Count", "True Labels (freq)", "ZS Conf (avg)"], 1):
        c = ws_s.cell(row=row_cursor, column=col, value=h)
        c.font = FONT_HDR; c.fill = FILL_HDR; c.alignment = ALIGN_C
        ws_s.column_dimensions[get_column_letter(col)].width = 25
    row_cursor += 1
    zs_by_caption = defaultdict(list)
    for r in zs_sorted:
        zs_by_caption[r["zs_pred"]].append(r)
    for cap in zs_class_order:
        recs = zs_by_caption.get(cap, [])
        cnt  = len(recs)
        if cnt == 0:
            continue
        from collections import Counter
        true_freq = Counter(r["true_label"] for r in recs)
        freq_str  = ", ".join(f"{k}:{v}" for k, v in true_freq.most_common())
        zsc       = f"{sum(r['zs_conf'] for r in recs)/cnt:.1%}"
        ws_s.cell(row=row_cursor, column=1, value=cap).font  = FONT_BODY
        ws_s.cell(row=row_cursor, column=2, value=cnt).font  = FONT_BODY
        ws_s.cell(row=row_cursor, column=3, value=freq_str).font = FONT_BODY
        ws_s.cell(row=row_cursor, column=4, value=zsc).font  = FONT_BODY
        row_cursor += 1

    wb.save(output_path)
    print(f"\n✅  Excel report saved → {output_path}")


print("Updated helpers defined ✅")


Updated helpers defined ✅


In [51]:
import torch
from typing import Tuple, Dict, Any, List
import torch.nn.functional as F
import numpy as np
import time
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

class ANNClassifier:
    def __init__(self, input_dim=None, hidden_dim1=224, hidden_dim2=128, C=1.0, max_iter=100, verbose=False, random_state=42):
        self.C = C
        self.loss_func = torch.nn.CrossEntropyLoss()
        self.max_iter = max_iter
        self.random_state = random_state
        self.verbose = verbose
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = input_dim  # Will be set during fit() if None
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.model = None

    def compute_loss(self, preds, labels):
        loss = self.loss_func(preds, labels)
        wreg = 0.5 * sum((param.norm(p=2) for param in self.model.parameters()))  # L2 regularization
        return loss.mean() + (1.0 / self.C) * wreg

    def predict_proba(self, feats):
        feats = feats.to(self.device)
        with torch.no_grad():
            return torch.softmax(self.model(feats), dim=1)

    def fit(self, feats, labels):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        # Set input_dim if not set
        if self.input_dim is None:
            self.input_dim = feats.shape[1]

        self.model = torch.nn.Sequential(
            torch.nn.Linear(self.input_dim, self.hidden_dim1),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim1, self.hidden_dim2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(self.hidden_dim2, 9)  # Assuming 9 classes
        ).to(self.device)

        for layer in self.model:
            if isinstance(layer, torch.nn.Linear):
                torch.nn.init.xavier_uniform_(layer.weight)

        feats = feats.to(self.device)
        labels = labels.long().to(self.device)

        opt = torch.optim.Adam(self.model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=5)

        min_loss = np.inf
        patience = 10
        patience_counter = 0

        for epoch in range(self.max_iter):
            opt.zero_grad()
            preds = self.model(feats)
            loss = self.compute_loss(preds, labels)
            loss.backward()
            opt.step()
            scheduler.step(loss)

            if self.verbose and epoch % 10 == 0:
                print(f"Epoch {epoch}: Loss: {loss:.4f}")

            if loss < min_loss:
                min_loss = loss
                patience_counter = 0
                best_model = self.model.state_dict()
            else:
                patience_counter += 1

            # if patience_counter >= patience:
            #     print(f"Early stopping at epoch {epoch}")
            #     break

        self.model.load_state_dict(best_model)

        if self.verbose:
            print(f"Final Loss: {min_loss:.4f}")

In [52]:
import json

with open(r'D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\best_parameters\best_params_Hoptimus.json', 'r') as f:
    best_params = json.load(f)

print("✅ Loaded best_params from best_params_Hoptimus.json",best_params)

✅ Loaded best_params from best_params_Hoptimus.json {'hidden_dim1': 256, 'hidden_dim2': 128, 'C': 10.0, 'max_iter': 500}


## Run the Full Pipeline

In [53]:
# ── CONCH feature extractor (mirrors H-optimus exactly) ──────────────────────
 
def load_conch_model(checkpoint_path, device):
    """Load CONCH ViT-B-16 from a local checkpoint."""
    from conch.open_clip_custom import create_model_from_pretrained
    print(f"  Loading CONCH model from {checkpoint_path} …")
    model, preprocess = create_model_from_pretrained(
        model_cfg       = "conch_ViT-B-16",
        checkpoint_path = checkpoint_path,
        device          = device,
        force_image_size= 224,
    )
    model.eval()
    return model, preprocess
 
 
def extract_conch_features(df, checkpoint_path, cache_path, device,
                            force=False, _model_tuple=None):
    """
    Extract L2-normalised CONCH image embeddings for every patch in df.
    Results are cached to cache_path — identical contract to extract_hoptimus_features.
    Pass _model_tuple=(model, preprocess) to reuse an already-loaded model.
    Returns (features [N, D], labels [N], paths [N], names [N]).
    """
    if not force and os.path.exists(cache_path):
        print(f"  Loading cached CONCH features: {cache_path}")
        cache = torch.load(cache_path, weights_only=False)
        return cache["features"], cache["labels"], cache["paths"], cache["names"]
 
    if _model_tuple is not None:
        model, preprocess = _model_tuple
    else:
        model, preprocess = load_conch_model(checkpoint_path, device)
 
    features, labels, paths, names = [], [], [], []
    with torch.inference_mode():
        for _, row in tqdm(df.iterrows(), total=len(df),
                           desc="Extracting CONCH features"):
            try:
                image  = Image.open(row["full_path"]).convert("RGB")
                tensor = preprocess(image).unsqueeze(0).to(device)
                feat   = model.encode_image(tensor)
                feat   = feat / feat.norm(dim=-1, keepdim=True)   # L2 normalise
                features.append(feat.cpu().squeeze(0))
                labels.append(row["label"])
                paths.append(row["full_path"])
                names.append(row["patch_name"])
            except Exception as e:
                print(f"  [WARN] skipping {row['full_path']}: {e}")
 
    X = torch.stack(features)
    y = np.array(labels)
    os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
    torch.save({"features": X, "labels": y,
                "paths": paths, "names": names}, cache_path)
    print(f"  Saved CONCH cache → {cache_path}")
 
    if _model_tuple is None:
        del model; torch.cuda.empty_cache()
    return X, y, paths, names

# ── Main inference cell ───────────────────────────────────────────────────────
 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")
 
# ── 1. Class list ────────────────────────────────────────────────────────────
with open(CFG["prompt_file"]) as f:
    prompt_data = json.load(f)["0"]
classnames_dict = prompt_data["classnames"]
classes_list    = list(classnames_dict.keys())
print(f"Classes from JSON ({len(classes_list)}): {classes_list}")
 
# ── Separate class lists for each classifier ─────────────────────────────
ZS_CLASS_ORDER = classes_list   # 14 classes from JSON (used by Zero-Shot & Few-Shot)
# ANN_CLASS_ORDER is already defined in the CFG cell (9 classes)

# Shared classes (by name) — used to build the dataset so every patch
# can be scored by BOTH classifiers.  Preserves JSON key order.
SHARED_CLASSES  = [c for c in ZS_CLASS_ORDER if c in ANN_CLASS_ORDER]
ZS_ONLY_CLASSES = [c for c in ZS_CLASS_ORDER if c not in ANN_CLASS_ORDER]
ANN_ONLY_CLASSES= [c for c in ANN_CLASS_ORDER if c not in ZS_CLASS_ORDER]

print(f"ZS  class list  ({len(ZS_CLASS_ORDER):>2}): {ZS_CLASS_ORDER}")
print(f"ANN class list  ({len(ANN_CLASS_ORDER):>2}): {ANN_CLASS_ORDER}")
print(f"Shared classes  ({len(SHARED_CLASSES):>2}): {SHARED_CLASSES}")
print(f"ZS-only classes ({len(ZS_ONLY_CLASSES):>2}): {ZS_ONLY_CLASSES}")
print(f"ANN-only classes({len(ANN_ONLY_CLASSES):>2}): {ANN_ONLY_CLASSES}")
print()
print("ℹ️  Dataset will be built from SHARED classes only so both")
print("   classifiers can be compared on every patch.")
print("   match = (zs_pred == ann_pred) — compared by class name string.")
 
# ── TEST_ONLY mode flag ──────────────────────────────────────────────────────
TEST_ONLY = int(CFG.get("test_only", 0))
if TEST_ONLY:
    print("⚠️  TEST_ONLY=1 — train cache will be skipped; inference on test set only.")
 
# ── 2. Build DataFrames for both folders ─────────────────────────────────────
# Use the union of all known class folders so every patch is included.
# ZS and ANN each use their own class list for prediction — no filtering needed.
ALL_FOLDER_CLASSES = sorted(set(ZS_CLASS_ORDER) | set(ANN_CLASS_ORDER))
print(f"\nScanning image folders (all {len(ALL_FOLDER_CLASSES)} known class folders) …")
if not TEST_ONLY:
    train_df = build_df_from_subfolders(CFG["train_image_folder"], ALL_FOLDER_CLASSES)
else:
    train_df = None   # never used in TEST_ONLY mode
test_df  = build_df_from_subfolders(CFG["test_image_folder"],  ALL_FOLDER_CLASSES)
if not TEST_ONLY:
    print(f"  Train folder patches : {len(train_df)}")
print(f"  Test  folder patches : {len(test_df)}")
 
# ── 3. H-optimus-1 feature extraction ────────────────────────────────────────
print("\nH-optimus-1 feature extraction …")
_need_hop_train = (not TEST_ONLY) and (
    CFG["force_extract"] or not os.path.exists(CFG["hoptimus_train_cache"]))
_need_hop_test  = CFG["force_extract"] or not os.path.exists(CFG["hoptimus_test_cache"])
if _need_hop_train or _need_hop_test:
    _hop_model = load_hoptimus_model(CFG["hoptimus_model_id"], CFG["hf_token"], device)
else:
    _hop_model = None
 
if not TEST_ONLY:
    print("  → Train folder")
    X_hop_train, y_hop_train, hop_train_paths, hop_train_names = extract_hoptimus_features(
        train_df,
        model_id   = CFG["hoptimus_model_id"],
        hf_token   = CFG["hf_token"],
        cache_path = CFG["hoptimus_train_cache"],
        device     = device,
        force      = CFG["force_extract"],
        _model     = _hop_model,
    )
    n_train = len(y_hop_train)
else:
    n_train = 0   # train arrays never allocated
 
print("  → Test folder")
X_hop_test, y_hop_test, hop_test_paths, hop_test_names = extract_hoptimus_features(
    test_df,
    model_id   = CFG["hoptimus_model_id"],
    hf_token   = CFG["hf_token"],
    cache_path = CFG["hoptimus_test_cache"],
    device     = device,
    force      = CFG["force_extract"],
    _model     = _hop_model,
)
if _hop_model is not None:
    del _hop_model; torch.cuda.empty_cache()
 
n_test = len(y_hop_test)
N      = n_train + n_test
print(f"  H-optimus: {n_train} train + {n_test} test = {N} total")
 
# ── 4. CONCH feature extraction ──────────────────────────────────────────────
print("\nCONCH feature extraction …")
_need_conch_train = (not TEST_ONLY) and (
    CFG["force_extract"] or not os.path.exists(CFG["conch_train_cache"]))
_need_conch_test  = CFG["force_extract"] or not os.path.exists(CFG["conch_test_cache"])
if _need_conch_train or _need_conch_test:
    _conch_tuple = load_conch_model(CFG["conch_checkpoint"], device)
else:
    _conch_tuple = None   # caches exist; loader will read from disk
 
if not TEST_ONLY:
    print("  → Train folder")
    X_conch_train, y_conch_train, conch_train_paths, conch_train_names = extract_conch_features(
        train_df,
        checkpoint_path = CFG["conch_checkpoint"],
        cache_path      = CFG["conch_train_cache"],
        device          = device,
        force           = CFG["force_extract"],
        _model_tuple    = _conch_tuple,
    )
 
print("  → Test folder")
X_conch_test, y_conch_test, conch_test_paths, conch_test_names = extract_conch_features(
    test_df,
    checkpoint_path = CFG["conch_checkpoint"],
    cache_path      = CFG["conch_test_cache"],
    device          = device,
    force           = CFG["force_extract"],
    _model_tuple    = _conch_tuple,
)
if _conch_tuple is not None:
    del _conch_tuple; torch.cuda.empty_cache()
 
if TEST_ONLY:
    print(f"  CONCH: (train skipped) + {len(y_conch_test)} test")
else:
    print(f"  CONCH: {len(y_conch_train)} train + {len(y_conch_test)} test")
 
# ── 5. Size sanity checks ─────────────────────────────────────────────────────
if not TEST_ONLY:
    assert len(X_conch_train) == n_train, (
        f"CONCH train cache has {len(X_conch_train)} entries but H-optimus train has {n_train}."
    )
assert len(X_conch_test) == n_test, (
    f"CONCH test cache has {len(X_conch_test)} entries but H-optimus test has {n_test}."
)
 
# Filename alignment checks (per folder)
def _check_alignment(names_a, label_a, names_b, label_b):
    if list(names_a) != list(names_b):
        raise RuntimeError(
            f"{label_a} and {label_b} caches have different filename orders!\n"
            "Re-extract both with force_extract=True."
        )
    print(f"  ✅ {label_a} and {label_b} filenames aligned")
 
if not TEST_ONLY:
    _check_alignment(conch_train_names, "CONCH-train",
                     hop_train_names,   "H-optimus-train")
_check_alignment(conch_test_names,  "CONCH-test",
                 hop_test_names,    "H-optimus-test")
 
# ── 6. Load / fit Logistic Regression ────────────────────────────────────────
if os.path.exists(CFG["lr_model_path"]):
    print(f"\nLoading saved LR model from {CFG['lr_model_path']} …")
    lr_clf = joblib.load(CFG["lr_model_path"])
elif TEST_ONLY:
    raise RuntimeError(
        "TEST_ONLY=1 but no saved LR model found at CFG['lr_model_path'].\n"
        "Either set TEST_ONLY=0 to fit on train features, or point lr_model_path "
        "to an existing .pkl file."
    )
else:
    print("\nNo saved LR model — fitting on CONCH train features …")
    lr_clf = fit_linear_probe(X_conch_train.numpy(), y_conch_train, CFG["random_seed"])
    os.makedirs(os.path.dirname(os.path.abspath(CFG["lr_model_path"])), exist_ok=True)
    joblib.dump(lr_clf, CFG["lr_model_path"])
    print(f"  LR model saved → {CFG['lr_model_path']}")
 
# ── 7. Load Zero-Shot weights ─────────────────────────────────────────────────
print("\nLoading Zero-Shot weights …")
zs_weights = torch.load(CFG["zs_weights_path"], map_location=device, weights_only=False)
 
# ── 8. Load ANN ───────────────────────────────────────────────────────────────
print("\nLoading ANN model …")
ann_model = ANNClassifier(
    input_dim   = 1536,
    hidden_dim1 = best_params["hidden_dim1"],
    hidden_dim2 = best_params["hidden_dim2"],
    C           = best_params["C"],
    max_iter    = best_params.get("max_iter", 100),
    verbose     = False,
)
ann_model.model = torch.load(CFG["ann_model_path"], map_location=device)
ann_model.model.to(device)
ann_model.model.eval()
print("  ANN loaded ✅")
 
# ── 9. Condition-based inference loop ────────────────────────────────────────
#
# Targets (per true-class for diff/same; per ZS-caption for caption sheet):
#   DIFFERENT sheet : 5-10 samples per TRUE class  (9 true classes)
#   SAME sheet      : 2-3  samples per TRUE class  (9 true classes)
#   ZS Captions     : 5-10 samples per ZS caption  (14 ZS captions)
#
# Each global index i in [0, N) maps to:
#   i  < n_train  →  train folder, local index = i
#   i >= n_train  →  test  folder, local index = i - n_train

DIFF_PER_CLASS_MIN = 5
DIFF_PER_CLASS_MAX = 10
SAME_PER_CLASS_MIN = 2
SAME_PER_CLASS_MAX = 3
ZS_CAP_PER_CAPTION_MIN = 5
ZS_CAP_PER_CAPTION_MAX = 10

print(f"\nRunning condition-based inference …")
print(f"  DIFFERENT: {DIFF_PER_CLASS_MIN}-{DIFF_PER_CLASS_MAX} per true class × {len(ANN_CLASS_ORDER)} classes")
print(f"  SAME     : {SAME_PER_CLASS_MIN}-{SAME_PER_CLASS_MAX} per true class × {len(ANN_CLASS_ORDER)} classes")
print(f"  ZS caps  : {ZS_CAP_PER_CAPTION_MIN}-{ZS_CAP_PER_CAPTION_MAX} per ZS caption × {len(ZS_CLASS_ORDER)} captions")

from collections import defaultdict

# Accumulators: dict[class_name] → list[row]
diff_by_class   = defaultdict(list)   # keyed by true_label
same_by_class   = defaultdict(list)   # keyed by true_label
zs_cap_by_pred  = defaultdict(list)   # keyed by zs_pred (the 14-caption class)

rng     = random.Random(CFG["random_seed"])
indices = list(range(N))
rng.shuffle(indices)

def _all_done():
    """Return True when every bucket has reached its maximum."""
    diff_done = all(len(diff_by_class[c]) >= DIFF_PER_CLASS_MAX for c in ANN_CLASS_ORDER)
    same_done = all(len(same_by_class[c]) >= SAME_PER_CLASS_MAX for c in ANN_CLASS_ORDER)
    zs_done   = all(len(zs_cap_by_pred[c]) >= ZS_CAP_PER_CAPTION_MAX for c in ZS_CLASS_ORDER)
    return diff_done and same_done and zs_done

for global_idx in indices:
    if _all_done():
        break

    # ── Resolve folder and local index ─────────────────────────────────────
    if (not TEST_ONLY) and global_idx < n_train:
        local_idx  = global_idx
        hop_feat   = X_hop_train[local_idx]
        conch_feat = X_conch_train[local_idx]
        true_label = y_hop_train[local_idx]
        full_path  = hop_train_paths[local_idx]
        filename   = hop_train_names[local_idx]
    else:
        local_idx  = global_idx if TEST_ONLY else global_idx - n_train
        hop_feat   = X_hop_test[local_idx]
        conch_feat = X_conch_test[local_idx]
        true_label = y_hop_test[local_idx]
        full_path  = hop_test_paths[local_idx]
        filename   = hop_test_names[local_idx]

    # Only collect patches whose true_label is one of the 9 ANN classes
    if true_label not in ANN_CLASS_ORDER:
        continue

    # ── Zero-Shot (CONCH features) ──────────────────────────────────────────
    cf       = conch_feat.unsqueeze(0).to(device)
    zs_logit = (100.0 * cf @ zs_weights).cpu()
    zs_prob  = torch.softmax(zs_logit, dim=-1)
    zs_pred  = ZS_CLASS_ORDER[zs_logit.argmax(dim=-1).item()]
    zs_conf  = float(zs_prob.max().item())

    # ── ANN (H-optimus features) ────────────────────────────────────────────
    hf = hop_feat.unsqueeze(0).to(device)
    with torch.inference_mode():
        ann_out = ann_model.model(hf).cpu()

    ann_pred_idx = ann_out.argmax(dim=-1).item()
    ann_pred     = ANN_CLASS_ORDER[ann_pred_idx]

    if CFG.get("ann_output_is_logprob", False):
        ann_prob = torch.exp(ann_out)
    else:
        ann_prob = torch.softmax(ann_out, dim=-1)

    ann_conf = float(ann_prob[0, ann_pred_idx].item())

    # ── Few-Shot LR (CONCH features) ───────────────────────────────────────
    _cf_np  = conch_feat.unsqueeze(0).numpy()
    fs_pred = lr_clf.predict(_cf_np)[0]
    fs_prob = lr_clf.predict_proba(_cf_np)
    fs_conf = float(fs_prob.max())

    match = (zs_pred == ann_pred)

    row = dict(
        match      = match,
        filename   = filename,
        full_path  = full_path,
        true_label = true_label,
        zs_pred    = zs_pred,
        ann_pred   = ann_pred,
        fs_pred    = fs_pred,
        zs_conf    = zs_conf,
        ann_conf   = ann_conf,
        fs_conf    = fs_conf,
    )

    # ── Route into DIFFERENT bucket (per true class, cap at MAX) ───────────
    if (not match
            and len(diff_by_class[true_label]) < DIFF_PER_CLASS_MAX):
        diff_by_class[true_label].append(row)

    # ── Route into SAME bucket (per true class, cap at MAX) ────────────────
    if (match
            and len(same_by_class[true_label]) < SAME_PER_CLASS_MAX):
        same_by_class[true_label].append(row)

    # ── Route into ZS-caption bucket (per zs_pred, cap at MAX) ────────────
    # This is independent of match/diff — any patch qualifies
    if len(zs_cap_by_pred[zs_pred]) < ZS_CAP_PER_CAPTION_MAX:
        zs_cap_by_pred[zs_pred].append(row)

# ── Flatten accumulators ───────────────────────────────────────────────────
diff_rows      = [r for cls in ANN_CLASS_ORDER for r in diff_by_class[cls]]
same_rows      = [r for cls in ANN_CLASS_ORDER for r in same_by_class[cls]]
zs_caption_rows= [r for cap in ZS_CLASS_ORDER  for r in zs_cap_by_pred[cap]]

print(f"\n  Collected — Different (ZS≠ANN): {len(diff_rows)} | Same (ZS=ANN): {len(same_rows)}")
print(f"  ZS Caption patches: {len(zs_caption_rows)}")

# ── Per-bucket warnings ────────────────────────────────────────────────────
print("\n  Per-class counts — DIFFERENT sheet:")
for cls in ANN_CLASS_ORDER:
    cnt = len(diff_by_class[cls])
    status = "✅" if cnt >= DIFF_PER_CLASS_MIN else f"⚠️  only {cnt}"
    print(f"    {cls:>6}: {cnt:>3}  {status}")

print("\n  Per-class counts — SAME sheet:")
for cls in ANN_CLASS_ORDER:
    cnt = len(same_by_class[cls])
    status = "✅" if cnt >= SAME_PER_CLASS_MIN else f"⚠️  only {cnt}"
    print(f"    {cls:>6}: {cnt:>3}  {status}")

print("\n  Per-caption counts — ZS Captions sheet:")
for cap in ZS_CLASS_ORDER:
    cnt = len(zs_cap_by_pred[cap])
    status = "✅" if cnt >= ZS_CAP_PER_CAPTION_MIN else f"⚠️  only {cnt}"
    print(f"    {cap:>10}: {cnt:>3}  {status}")

# ── 10. Write Excel report ────────────────────────────────────────────────────
write_excel(
    diff_rows       = diff_rows,
    same_rows       = same_rows,
    zs_caption_rows = zs_caption_rows,
    output_path     = CFG["output_xlsx"],
    thumb_size      = CFG["thumb_size"],
    ann_class_order = ANN_CLASS_ORDER,
    zs_class_order  = ZS_CLASS_ORDER,
)
print("Done! 🎉")


Device: cuda:0

Classes from JSON (14): ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM', 'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']
ZS  class list  (14): ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM', 'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']
ANN class list  ( 9): ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
Shared classes  ( 7): ['ADI', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR']
ZS-only classes ( 7): ['PLC', 'LYA', 'ADE', 'MES', 'CAR', 'PDC', 'SIG']
ANN-only classes( 2): ['BACK', 'TUM']

ℹ️  Dataset will be built from SHARED classes only so both
   classifiers can be compared on every patch.
   match = (zs_pred == ann_pred) — compared by class name string.

Scanning image folders (all 16 known class folders) …
  [WARN] class folder not found: D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM\ADE
  [WARN] class folder not found: D:\Aamir Gulzar\dataset\CRC100K\NCT-CRC-HE-100K-NONORM\CAR
  [WARN] class folder not found: D:\Aamir Gulzar\datase

## Sanity checks for ann model

In [54]:
"""
ANN Diagnostic v3 — Label alignment + logit distribution check
===============================================================
The v2 output showed near-uniform confidences (~0.13) across ALL patches,
which means the model is not recognising ANY of the input features.
This script checks whether the H-optimus cache features are actually
compatible with what the model was trained on.

Edit CONFIG then run:
    python ann_inference_debug_v3.py
"""

import torch
import torch.nn as nn
import numpy as np

# ── CONFIG ────────────────────────────────────────────────────────────────────
ANN_MODEL_PATH      = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\TissueClassifier_CRC100K\models\ann_crc100k_Hoptimus.pth"
HOPTIMUS_CACHE_PATH = r"Cache\hoptimus_test_features.pt"

# This must match EXACTLY the order used during ANN training
# (the order classes were mapped to integers 0-8)
ANN_CLASS_ORDER = ["ADI", "BACK", "DEB", "LYM", "MUC", "MUS", "NORM", "STR", "TUM"]

# Also point at the TRAIN cache if you have it — helps confirm feature scale
HOPTIMUS_TRAIN_CACHE_PATH = None  # e.g. r"Cache\hoptimus_train_features.pt"
# ─────────────────────────────────────────────────────────────────────────────


def load_model(path, device):
    model = torch.load(path, map_location=device, weights_only=False)
    model.to(device)
    model.eval()
    return model


def run():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")

    model = load_model(ANN_MODEL_PATH, device)

    cache = torch.load(HOPTIMUS_CACHE_PATH, map_location="cpu", weights_only=False)
    X = cache["features"]   # [N, D]
    y = cache["labels"]     # [N] — could be ints or strings
    names = cache.get("names", [f"patch_{i}" for i in range(len(X))])

    print(f"Cache: {X.shape[0]} patches, dim={X.shape[1]}")
    print(f"Label dtype : {type(y[0]) if hasattr(y, '__getitem__') else type(y)}")
    print(f"Label sample: {y[:5]}")
    print(f"Label unique: {np.unique(y)}")

    # ── 1. Feature scale check ────────────────────────────────────────────
    print(f"\n── Feature scale check ─────────────────────────────────────────")
    norms = X.norm(dim=-1)
    print(f"  L2 norm — mean: {norms.mean():.4f}  std: {norms.std():.4f}  "
          f"min: {norms.min():.4f}  max: {norms.max():.4f}")
    if abs(norms.mean().item() - 1.0) < 0.05:
        print("  ✅ Features are L2-normalised (norm ≈ 1.0) — scale is correct.")
    else:
        print("  ⚠️  Features are NOT L2-normalised.")
        print("     The ANN was trained on H-optimus features. Check whether")
        print("     L2 normalisation was applied during training but not here,")
        print("     or vice versa.")

    # ── 2. Full-dataset forward pass ──────────────────────────────────────
    print(f"\n── Full forward pass on all {len(X)} patches ───────────────────")
    BATCH = 256
    all_logits = []
    with torch.inference_mode():
        for start in range(0, len(X), BATCH):
            batch = X[start:start+BATCH].to(device)
            all_logits.append(model(batch).cpu())
    all_logits = torch.cat(all_logits, dim=0)   # [N, 9]
    all_probs  = torch.softmax(all_logits, dim=-1)
    all_preds  = all_logits.argmax(dim=-1).numpy()

    # ── 3. Logit distribution — the key diagnostic ────────────────────────
    print(f"\n── Logit statistics per output neuron ──────────────────────────")
    print(f"  {'Neuron':>8}  {'Class':>6}  {'Mean':>8}  {'Std':>8}  {'Min':>8}  {'Max':>8}")
    for i, cls in enumerate(ANN_CLASS_ORDER):
        col = all_logits[:, i]
        print(f"  {i:>8}  {cls:>6}  {col.mean():>8.4f}  {col.std():>8.4f}  "
              f"{col.min():>8.4f}  {col.max():>8.4f}")

    overall_std = all_logits.std().item()
    print(f"\n  Overall logit std across all neurons & patches: {overall_std:.5f}")
    if overall_std < 0.05:
        print("  🐛 VERY LOW logit variance — the model is not responding to the input.")
        print("     Most likely cause: the features fed to the model are wrong.")
        print("     See 'Root cause checklist' printed below.")
    else:
        print("  ✅ Logit variance looks healthy.")

    # ── 4. Prediction distribution ────────────────────────────────────────
    print(f"\n── Prediction distribution (what the model actually predicts) ───")
    pred_counts = np.bincount(all_preds, minlength=9)
    for i, cls in enumerate(ANN_CLASS_ORDER):
        bar = "█" * int(pred_counts[i] / len(X) * 40)
        print(f"  {cls:>6} [{i}]: {pred_counts[i]:>5}  {bar}")

    # ── 5. Accuracy with correct label mapping ────────────────────────────
    print(f"\n── Accuracy with label-to-int mapping ──────────────────────────")
    label_to_int = {cls: i for i, cls in enumerate(ANN_CLASS_ORDER)}

    # Handle string labels
    if isinstance(y[0], str):
        print("  Labels are strings — mapping via ANN_CLASS_ORDER …")
        try:
            y_int = np.array([label_to_int[lbl] for lbl in y])
            print("  ✅ All labels mapped successfully.")
        except KeyError as e:
            print(f"  🐛 Unknown label in cache: {e}")
            print(f"     Cache labels : {np.unique(y)}")
            print(f"     Expected     : {ANN_CLASS_ORDER}")
            return
    else:
        y_int = np.array(y, dtype=int)
        print("  Labels are already integers.")

    acc = np.mean(all_preds == y_int)
    print(f"\n  ✅ Accuracy (correct mapping): {acc*100:.2f}%")
    print(f"  Mean confidence             : {all_probs.max(dim=-1).values.mean():.4f}")

    # Per-class accuracy
    print(f"\n── Per-class accuracy ──────────────────────────────────────────")
    print(f"  {'Class':>6}  {'Correct':>8}  {'Total':>8}  {'Acc':>8}")
    for i, cls in enumerate(ANN_CLASS_ORDER):
        mask = y_int == i
        if mask.sum() == 0:
            continue
        cls_acc = np.mean(all_preds[mask] == i)
        print(f"  {cls:>6}  {int(cls_acc * mask.sum()):>8}  {int(mask.sum()):>8}  {cls_acc*100:>7.1f}%")

    # ── 6. Root cause checklist ───────────────────────────────────────────
    if overall_std < 0.05:
        print(f"\n── Root cause checklist ────────────────────────────────────────")
        print("  The model outputs near-uniform logits on every patch.")
        print("  Work through these in order:\n")
        print("  [ ] 1. WRONG CACHE")
        print("         Are you loading the H-optimus cache or the CONCH cache?")
        print("         The ANN was trained on H-optimus (dim=1536) features.")
        print("         CONCH is also dim=512 — dim=1536 alone doesn't confirm it's H-optimus.\n")
        print("  [ ] 2. L2 NORMALISATION MISMATCH")
        print("         Check your training notebook:")
        print("         Were H-optimus features L2-normalised before training the ANN?")
        print("         If yes, the cache must also be L2-normalised.")
        print("         If no, the cache must NOT be normalised.\n")
        print("  [ ] 3. WRONG MODEL FILE")
        print("         Is ann_crc100k_Hoptimus.pth the final best model,")
        print("         or an intermediate checkpoint from a failed run?\n")
        print("  [ ] 4. TRAIN CACHE vs TEST CACHE MISMATCH")
        if HOPTIMUS_TRAIN_CACHE_PATH:
            train_cache = torch.load(HOPTIMUS_TRAIN_CACHE_PATH, map_location="cpu", weights_only=False)
            X_train = train_cache["features"]
            train_norms = X_train.norm(dim=-1)
            test_norms  = norms
            print(f"         Train feature norm: {train_norms.mean():.4f} ± {train_norms.std():.4f}")
            print(f"         Test  feature norm: {test_norms.mean():.4f} ± {test_norms.std():.4f}")
            if abs(train_norms.mean() - test_norms.mean()) > 0.1:
                print("         🐛 MISMATCH — train and test features have different scales!")
            else:
                print("         ✅ Train and test feature norms match.")
        else:
            print("         Set HOPTIMUS_TRAIN_CACHE_PATH to compare train vs test feature scales.\n")


if __name__ == "__main__":
    run()

Device: cuda

Cache: 7180 patches, dim=1536
Label dtype : <class 'numpy.str_'>
Label sample: ['ADI' 'ADI' 'ADI' 'ADI' 'ADI']
Label unique: ['ADI' 'BACK' 'DEB' 'LYM' 'MUC' 'MUS' 'NORM' 'STR' 'TUM']

── Feature scale check ─────────────────────────────────────────
  L2 norm — mean: 1.0000  std: 0.0000  min: 1.0000  max: 1.0000
  ✅ Features are L2-normalised (norm ≈ 1.0) — scale is correct.

── Full forward pass on all 7180 patches ───────────────────

── Logit statistics per output neuron ──────────────────────────
    Neuron   Class      Mean       Std       Min       Max
         0     ADI    0.0027    0.0701   -0.0719    0.2035
         1    BACK   -0.0127    0.0463   -0.0782    0.1703
         2     DEB   -0.0076    0.0387   -0.0700    0.2258
         3     LYM   -0.0090    0.0492   -0.0549    0.1878
         4     MUC    0.0064    0.0658   -0.0647    0.2205
         5     MUS    0.0042    0.0514   -0.0522    0.2226
         6    NORM    0.0010    0.0584   -0.0677    0.2102
         

## Extended Excel Report for Zero shot analysis
  
Four sheets are produced:

| Sheet | Contents |
|---|---|
| **Fine-Grained (ZS → GT subtype)** | ≥ 10 examples per valid (GT, ZS-pred) pair where ZS predicted a biologically valid subclass of the GT |
| **Correct (ZS = GT)** | 5 examples per GT class where ZS prediction exactly matches the ground truth |
| **ZS Predictions (14 classes)** | 5–10 examples per ZS-predicted label — same logic as the original ZS Captions sheet |
| **Summary** | Patch counts, per-class breakdowns and per-caption breakdowns |

In [55]:
# ── Zero-Shot → Ground-Truth subtype mapping ─────────────────────────────────
ZS_SUBTYPE_MAP = {
    "ADI":  set(),
    "BACK": set(),
    "DEB":  set(),
    "LYM":  {"LYA"},
    "MUC":  set(),
    "MUS":  {"MES"},
    "NORM": set(),
    "STR":  {"MES"},
    "TUM":  {"CAR", "PDC", "ADE", "SIG"},
}

def _is_exact_match(gt, zs):
    return gt == zs

def _is_valid_subtype(gt, zs):
    return zs in ZS_SUBTYPE_MAP.get(gt, set())

# Note: FILL_ZS is already defined in your environment
FILL_SUB  = PatternFill("solid", fgColor="FFF0CC")
FILL_CORR = PatternFill("solid", fgColor="D0FFD0")

_HEADERS_EXT = [
    ("#",                 5),
    ("Patch Image",       22),
    ("Filename",          30),
    ("True Label",        16),
    ("ZS Pred",           16),
    ("ANN Pred",          16),
    ("Few-Shot Pred",     18),
    ("ZS Confidence",     14),
    ("ANN Confidence",    14),
    ("FS Confidence",     14),
]

def _embed_image_fixed(ws, row, col_letter, full_path, thumb_size):
    """
    Robust image embedding using unique BytesIO streams to prevent 
    drawing relation errors and workbook corruption.
    """
    try:
        with Image.open(full_path) as img_pil:
            img_pil = img_pil.convert("RGB")
            img_pil.thumbnail(thumb_size, Image.LANCZOS)
            
            # Save to a fresh buffer for every instance to avoid XML ID collisions
            buf = io.BytesIO()
            img_pil.save(buf, format="PNG")
            buf.seek(0)
            
            xl_img = XLImage(buf)
            ws.add_image(xl_img, f"{col_letter}{row}")
    except Exception as e:
        ws.cell(row=row, column=2, value=f"[img err: {e}]")

def _write_header_row(ws, headers):
    for col, (h, w) in enumerate(headers, 1):
        _cell(ws, 1, col, h, font=FONT_HDR, fill=FILL_HDR, align=ALIGN_C)
        ws.column_dimensions[get_column_letter(col)].width = w
    ws.row_dimensions[1].height = 22

def _write_extended_row(ws, row_num, r, fill_bg, thumb_size, index):
    ROW_H = thumb_size[1] + 14
    ws.row_dimensions[row_num].height = ROW_H

    _cell(ws, row_num, 1,  index,             FONT_BODY, fill_bg,    ALIGN_C)
    _cell(ws, row_num, 3,  r["filename"],     FONT_BODY, fill_bg,    ALIGN_L)
    _cell(ws, row_num, 4,  r["true_label"],   FONT_BODY, fill_bg,    ALIGN_C)
    _cell(ws, row_num, 5,  r["zs_pred"],      FONT_BODY, fill_bg,    ALIGN_C)
    _cell(ws, row_num, 6,  r["ann_pred"],     FONT_BODY, fill_bg,    ALIGN_C)
    _cell(ws, row_num, 7,  r["fs_pred"],      FONT_BODY, FILL_BONUS, ALIGN_C)
    _cell(ws, row_num, 8,  f"{r['zs_conf']:.1%}",   FONT_BODY, fill_bg,    ALIGN_C)
    _cell(ws, row_num, 9,  f"{r['ann_conf']:.1%}",  FONT_BODY, fill_bg,    ALIGN_C)
    _cell(ws, row_num, 10, f"{r['fs_conf']:.1%}",   FONT_BODY, FILL_BONUS, ALIGN_C)
    
    # Using the fixed embedding logic
    _embed_image_fixed(ws, row_num, "B", r["full_path"], thumb_size)

# ── Sheet writers ─────────────────────────────────────────────────────────────

def _write_subtype_sheet(ws, rows, thumb_size):
    _write_header_row(ws, _HEADERS_EXT)
    for i, r in enumerate(rows):
        _write_extended_row(ws, i + 2, r, FILL_SUB, thumb_size, i + 1)
    return ws

def _write_correct_sheet(ws, rows, thumb_size):
    _write_header_row(ws, _HEADERS_EXT)
    for i, r in enumerate(rows):
        _write_extended_row(ws, i + 2, r, FILL_CORR, thumb_size, i + 1)
    return ws

def _write_zs14_sheet(ws, rows, thumb_size):
    _write_header_row(ws, _HEADERS_EXT)
    for i, r in enumerate(rows):
        _write_extended_row(ws, i + 2, r, FILL_ZS, thumb_size, i + 1)
    return ws

def _write_summary_sheet_ext(ws, correct_rows, subtype_rows, zs14_rows,
                             ann_class_order, zs_class_order):
    from collections import Counter, defaultdict

    ws.column_dimensions["A"].width = 34
    ws.column_dimensions["B"].width = 14
    ws.column_dimensions["C"].width = 14
    ws.column_dimensions["D"].width = 14
    ws.column_dimensions["E"].width = 24

    ws["A1"] = "Extended ZS Analysis — Summary"
    ws["A1"].font = Font(bold=True, size=13, name="Arial")

    summary_stats = [
        ("Correct patches (ZS = GT)",            len(correct_rows)),
        ("Fine-Grained patches (ZS → GT subtype)", len(subtype_rows)),
        ("ZS Predictions sheet patches",            len(zs14_rows)),
        ("Grand total",                             len(correct_rows) + len(subtype_rows)),
    ]
    for r_off, (lbl, val) in enumerate(summary_stats, 3):
        ws.cell(row=r_off, column=1, value=lbl).font = FONT_BODY
        ws.cell(row=r_off, column=2, value=val).font = FONT_BODY

    # ── Section A: Correct sheet breakdown ───────────────────────────────────
    cur = 9
    ws.cell(row=cur, column=1, value="Per-class breakdown — Correct (ZS = GT)").font = Font(bold=True, name="Arial")
    cur += 1
    for col, h in enumerate(["GT Class", "Count", "ZS Conf (avg)", "ANN Conf (avg)", "FS Conf (avg)"], 1):
        c = ws.cell(row=cur, column=col, value=h)
        c.font = FONT_HDR; c.fill = FILL_HDR; c.alignment = ALIGN_C
    cur += 1
    corr_by_cls = defaultdict(list)
    for r in correct_rows:
        corr_by_cls[r["true_label"]].append(r)
    for cls in ann_class_order:
        recs = corr_by_cls.get(cls, [])
        cnt  = len(recs)
        zsc  = f"{sum(x['zs_conf']  for x in recs)/cnt:.1%}" if cnt else "—"
        ac   = f"{sum(x['ann_conf'] for x in recs)/cnt:.1%}" if cnt else "—"
        fc   = f"{sum(x['fs_conf']  for x in recs)/cnt:.1%}" if cnt else "—"
        ws.cell(row=cur, column=1, value=cls).font = FONT_BODY
        ws.cell(row=cur, column=2, value=cnt).font = FONT_BODY
        ws.cell(row=cur, column=3, value=zsc).font = FONT_BODY
        ws.cell(row=cur, column=4, value=ac).font  = FONT_BODY
        ws.cell(row=cur, column=5, value=fc).font  = FONT_BODY
        cur += 1

    # ── Section B: Fine-Grained breakdown ────────────────────────────────────
    cur += 1
    ws.cell(row=cur, column=1, value="Per-(GT, ZS) pair breakdown — Fine-Grained").font = Font(bold=True, name="Arial")
    cur += 1
    for col, h in enumerate(["GT Class", "ZS Pred", "Count", "ZS Conf (avg)", "ANN Conf (avg)"], 1):
        c = ws.cell(row=cur, column=col, value=h)
        c.font = FONT_HDR; c.fill = FILL_HDR; c.alignment = ALIGN_C
    cur += 1
    sub_by_pair = defaultdict(list)
    for r in subtype_rows:
        sub_by_pair[(r["true_label"], r["zs_pred"])].append(r)
    for gt in ann_class_order:
        for zs in sorted(ZS_SUBTYPE_MAP.get(gt, set())):
            recs = sub_by_pair.get((gt, zs), [])
            cnt  = len(recs)
            if cnt == 0: continue
            zsc = f"{sum(x['zs_conf']  for x in recs)/cnt:.1%}"
            ac  = f"{sum(x['ann_conf'] for x in recs)/cnt:.1%}"
            ws.cell(row=cur, column=1, value=gt).font  = FONT_BODY
            ws.cell(row=cur, column=2, value=zs).font  = FONT_BODY
            ws.cell(row=cur, column=3, value=cnt).font = FONT_BODY
            ws.cell(row=cur, column=4, value=zsc).font = FONT_BODY
            ws.cell(row=cur, column=5, value=ac).font  = FONT_BODY
            cur += 1

    # ── Section C: ZS-14 caption breakdown ───────────────────────────────────
    cur += 1
    ws.cell(row=cur, column=1, value="Per-ZS-label breakdown — ZS Predictions (14 classes)").font = Font(bold=True, name="Arial")
    cur += 1
    for col, h in enumerate(["ZS Pred", "Count", "GT labels (freq)", "ZS Conf (avg)"], 1):
        c = ws.cell(row=cur, column=col, value=h)
        c.font = FONT_HDR; c.fill = FILL_HDR; c.alignment = ALIGN_C
    cur += 1
    zs14_by_pred = defaultdict(list)
    for r in zs14_rows:
        zs14_by_pred[r["zs_pred"]].append(r)
    for cap in zs_class_order:
        recs = zs14_by_pred.get(cap, [])
        cnt  = len(recs)
        if cnt == 0: continue
        freq_str = ", ".join(f"{k}:{v}" for k, v in Counter(r["true_label"] for r in recs).most_common())
        zsc = f"{sum(x['zs_conf'] for x in recs)/cnt:.1%}"
        ws.cell(row=cur, column=1, value=cap).font      = FONT_BODY
        ws.cell(row=cur, column=2, value=cnt).font      = FONT_BODY
        ws.cell(row=cur, column=3, value=freq_str).font = FONT_BODY
        ws.cell(row=cur, column=4, value=zsc).font      = FONT_BODY
        cur += 1
    return ws

# ── Sampling logic ────────────────────────────────────────────────────────────

CORRECT_PER_CLASS     = 5
SUBTYPE_PER_PAIR      = 10
ZS14_PER_CAPTION_MIN  = 5
ZS14_PER_CAPTION_MAX  = 10

THUMB_SIZE_EXT = (CFG["thumb_size"][0] + 30, CFG["thumb_size"][1] + 30)

print("Extended report — sampling patches …")
corr_by_class   = defaultdict(list)
sub_by_pair     = defaultdict(list)
zs14_by_pred    = defaultdict(list)

ALL_SUBTYPE_PAIRS = [(gt, zs) for gt, zs_set in ZS_SUBTYPE_MAP.items() for zs in zs_set]

def _all_done_ext():
    corr_done = all(len(corr_by_class[c]) >= CORRECT_PER_CLASS for c in ANN_CLASS_ORDER)
    sub_done  = all(len(sub_by_pair[(gt, zs)]) >= SUBTYPE_PER_PAIR for gt, zs in ALL_SUBTYPE_PAIRS)
    zs14_done = all(len(zs14_by_pred[c]) >= ZS14_PER_CAPTION_MAX for c in ZS_CLASS_ORDER)
    return corr_done and sub_done and zs14_done

rng2     = random.Random(CFG["random_seed"] + 1)
indices2 = list(range(N))
rng2.shuffle(indices2)

for global_idx in indices2:
    if _all_done_ext(): break

    if (not TEST_ONLY) and global_idx < n_train:
        l_idx = global_idx
        hf, cf, tl, fp, fn = X_hop_train[l_idx], X_conch_train[l_idx], y_hop_train[l_idx], hop_train_paths[l_idx], hop_train_names[l_idx]
    else:
        l_idx = global_idx if TEST_ONLY else global_idx - n_train
        hf, cf, tl, fp, fn = X_hop_test[l_idx], X_conch_test[l_idx], y_hop_test[l_idx], hop_test_paths[l_idx], hop_test_names[l_idx]

    if tl not in ANN_CLASS_ORDER: continue

    # Zero-Shot Inference
    with torch.inference_mode():
        cf_dev = cf.unsqueeze(0).to(device)
        zs_logit = (100.0 * cf_dev @ zs_weights).cpu()
        zs_prob  = torch.softmax(zs_logit, dim=-1)
        zs_idx   = zs_logit.argmax(dim=-1).item()
        zs_pred, zs_conf = ZS_CLASS_ORDER[zs_idx], float(zs_prob.max().item())

        # ANN Inference
        hf_dev = hf.unsqueeze(0).to(device)
        ann_out = ann_model.model(hf_dev).cpu()
        ann_idx = ann_out.argmax(dim=-1).item()
        ann_prob = torch.softmax(ann_out, dim=-1)
        ann_pred, ann_conf = ANN_CLASS_ORDER[ann_idx], float(ann_prob[0, ann_idx].item())

    # Few-Shot Inference
    cf_np   = cf.unsqueeze(0).numpy()
    fs_pred = lr_clf.predict(cf_np)[0]
    fs_conf = float(lr_clf.predict_proba(cf_np).max())

    row = dict(filename=fn, full_path=fp, true_label=tl, zs_pred=zs_pred, 
               ann_pred=ann_pred, fs_pred=fs_pred, zs_conf=zs_conf, 
               ann_conf=ann_conf, fs_conf=fs_conf)

    if _is_exact_match(tl, zs_pred) and len(corr_by_class[tl]) < CORRECT_PER_CLASS:
        corr_by_class[tl].append(row)
    if _is_valid_subtype(tl, zs_pred) and len(sub_by_pair[(tl, zs_pred)]) < SUBTYPE_PER_PAIR:
        sub_by_pair[(tl, zs_pred)].append(row)
    if len(zs14_by_pred[zs_pred]) < ZS14_PER_CAPTION_MAX:
        zs14_by_pred[zs_pred].append(row)

# Flatten
correct_rows = [r for cls in ANN_CLASS_ORDER for r in corr_by_class[cls]]
subtype_rows = [r for (gt, zs) in ALL_SUBTYPE_PAIRS for r in sub_by_pair[(gt, zs)]]
zs14_rows    = [r for cap in ZS_CLASS_ORDER for r in zs14_by_pred[cap]]

# ── Build & save workbook ─────────────────────────────────────────────────────
_base, _ext = os.path.splitext(CFG["output_xlsx"])
EXTENDED_XLSX = _base + "_extended" + _ext
os.makedirs(os.path.dirname(os.path.abspath(EXTENDED_XLSX)), exist_ok=True)

from openpyxl import Workbook as _WB
wb_ext = _WB()

# Setup sheets
ws_sub = wb_ext.active
ws_sub.title = "Fine-Grained"
ws_corr = wb_ext.create_sheet("Correct (ZS = GT)")
ws_zs14 = wb_ext.create_sheet("Extra ZS Predictions")
ws_summ = wb_ext.create_sheet("Summary")

# Populate
_write_subtype_sheet(ws_sub,  subtype_rows, THUMB_SIZE_EXT)
_write_correct_sheet(ws_corr, correct_rows, THUMB_SIZE_EXT)
_write_zs14_sheet(ws_zs14, zs14_rows,   THUMB_SIZE_EXT)
_write_summary_sheet_ext(ws_summ, correct_rows, subtype_rows, zs14_rows, ANN_CLASS_ORDER, ZS_CLASS_ORDER)

wb_ext.active = ws_summ 
wb_ext.save(EXTENDED_XLSX)
print(f"\n✅ Extended Excel report saved → {EXTENDED_XLSX}")

Extended report — sampling patches …

✅ Extended Excel report saved → Results\classifier_comparison_report_extended.xlsx
